In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import pandas as pd
import numpy as np

# 1. Locate and load the data
# (Double check your exact path on the right panel if this errors out!)
data_path = '/kaggle/input/india-runs-challenge-data/India_runs_data_and_ai_challenge/candidates.jsonl'
candidates = pd.read_json(data_path, lines=True)

# 2. Extract Features
def extract_advanced_features(df):
    features = []
    banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    
    for idx, row in df.iterrows():
        history = row['career_history'] if isinstance(row['career_history'], list) else []
        has_product_company = False
        is_pure_services = True if len(history) > 0 else False
        
        for job in history:
            company = str(job.get('company', '')).lower().strip()
            if company and not any(banned in company for banned in banned_companies):
                has_product_company = True
                is_pure_services = False
        
        skills_list = row['skills'] if isinstance(row['skills'], list) else []
        ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}
        core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).lower() for kw in ai_keywords))
                
        signals = row['redrob_signals'] if isinstance(row['redrob_signals'], dict) else {}
        completeness = signals.get('profile_completeness_score', 0)
        days_since_last_login = signals.get('days_since_last_login', 0)
        
        features.append({
            'candidate_id': row['candidate_id'],
            'completeness': completeness,
            'has_product_company': has_product_company,
            'is_pure_services': is_pure_services,
            'core_ai_skills': core_ai_skills_count,
            'days_since_last_login': days_since_last_login
        })
    return pd.DataFrame(features)

derived = extract_advanced_features(candidates)

# 3. Score & Break Ties
score = derived['completeness'] / 100.0 + derived['core_ai_skills'] * 0.15
score = np.where(derived['has_product_company'] == True, score + 0.25, score)
score = np.where(derived['is_pure_services'] == True, score - 0.40, score)
score += (100 - derived['days_since_last_login'].clip(0, 365)) / 500.0

normalized_score = 0.001 + (score - score.min()) * (0.999 - 0.001) / (score.max() - score.min())
derived['score'] = np.round(normalized_score, 4)

# 4. Sort and Format
derived = derived.sort_values(by='score', ascending=False).reset_index(drop=True)
derived['rank'] = derived.index + 1
derived['reasoning'] = derived.apply(lambda r: f"AI Engineer with {r['core_ai_skills']} core skills; background in {'Product Eng' if r['has_product_company'] else 'Services'}; platform activity stable.", axis=1)

submission = derived[['candidate_id', 'rank', 'score', 'reasoning']]

# 5. Export directly to Excel sheet
submission.to_excel('submission.xlsx', index=False)
print("SUCCESS: submission.xlsx has been created!")

/tmp/ipykernel_58/798836072.py:8: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  candidates = pd.read_json(data_path, lines=True)


ValueError: Expected object or value

In [3]:
import os
import pandas as pd
import numpy as np

# 1. Correctly locate the exact file path
# Let's search the input folder dynamically to find the correct path to candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please ensure the dataset is attached correctly.")
else:
    print(f"Found file at: {target_path}")
    
    # Load the data using the dynamically found path
    candidates = pd.read_json(target_path, lines=True)
    print("Successfully loaded candidates data!")

    # 2. Extract Features
    def extract_advanced_features(df):
        features = []
        banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
        
        for idx, row in df.iterrows():
            history = row['career_history'] if isinstance(row['career_history'], list) else []
            has_product_company = False
            is_pure_services = True if len(history) > 0 else False
            
            for job in history:
                company = str(job.get('company', '')).lower().strip()
                if company and not any(banned in company for banned in banned_companies):
                    has_product_company = True
                    is_pure_services = False
            
            skills_list = row['skills'] if isinstance(row['skills'], list) else []
            ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}
            core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).

SyntaxError: incomplete input (2098096174.py, line 41)

# MatchMind AI: Intelligent Candidate Discovery Pipeline

An end-to-end Data & AI solution designed to match candidate profiles against job descriptions using a two-stage semantic information retrieval framework.

## 🚀 The Architecture
To overcome the limitations of traditional keyword matching (which suffers from vocabulary mismatches and keyword-stuffing exploits), this project implements a **Two-Stage Retrieval & Ranking System**:

1. **Stage 1 (Semantic Filtering):** Uses a Bi-Encoder (`all-MiniLM-L6-v2`) to generate dense vector embeddings of job descriptions and candidate text, filtering down the dataset using rapid Cosine Similarity.
2. **Stage 2 (Deep Re-ranking):** Passes top candidate pairs through a Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) to analyze deep contextual alignments and eliminate candidates who rely on keyword-stuffing without practical project substance.

## 🛠️ How to Run on Kaggle or Locally

### Setup Dependencies
Install the required packages using pip:
```bash
pip install -r requirements.txt

In [4]:
import os
import pandas as pd
import numpy as np

# 1. Dynamically locate the exact file path to candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please make sure your dataset is attached.")
else:
    print(f"Found file at: {target_path}")
    
    # 2. Load the JSON Lines data properly
    print("Loading data... (this might take a moment for 100k rows)")
    candidates = pd.read_json(target_path, lines=True)
    print("Successfully loaded candidates data!")

    # 3. Extract Advanced Features
    print("Extracting features...")
    def extract_advanced_features(df):
        features = []
        banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
        
        for idx, row in df.iterrows():
            history = row['career_history'] if isinstance(row['career_history'], list) else []
            has_product_company = False
            is_pure_services = True if len(history) > 0 else False
            
            for job in history:
                company = str(job.get('company', '')).lower().strip()
                if company and not any(banned in company for banned in banned_companies):
                    has_product_company = True
                    is_pure_services = False
            
            skills_list = row['skills'] if isinstance(row['skills'], list) else []
            ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}
            core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).

SyntaxError: incomplete input (985722966.py, line 42)

In [6]:
pip install sentence-transformers pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Cell 1: Install libraries in the Kaggle environment
!pip install sentence-transformers pandas scikit-learn -q

# ==========================================
# Cell 2: Run the Two-Stage AI Matching Pipeline
# ==========================================
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Check if Kaggle's GPU accelerator is active
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 1. Load the Two-Stage Models onto the GPU
print("Loading semantic models...")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

# 2. Mock Data (Replace these arrays with your loaded Kaggle dataset DataFrames!)
job_description = "Looking for a Backend Developer with heavy experience in Python and FastAPI."
candidates = [
    {"id": "C001", "text": "Senior Python engineer. Expert in building asynchronous REST APIs using FastAPI."},
    {"id": "C002", "text": "Backend Developer. Pro in web systems. Technologies: Python, Django, Flask."},
    {"id": "C003", "text": "FastAPI FastAPI FastAPI Python Python Python Developer expert engineer."},
    {"id": "C004", "text": "Frontend Specialist. Expert in UI/UX architectures and React.js."}
]

# 3. Stage 1: Fast Filter (Bi-Encoder)
print("\n--- Running Stage 1: Retrieval ---")
jd_embedding = bi_encoder.encode(job_description).reshape(1, -1)
candidate_texts = [c["text"] for c in candidates]
candidate_embeddings = bi_encoder.encode(candidate_texts)

similarities = cosine_similarity(jd_embedding, candidate_embeddings)[0]
for i, candidate in enumerate(candidates):
    candidate["stage1_score"] = float(similarities[i])

# Take top candidates forward
top_candidates = sorted(candidates, key=lambda x: x["stage1_score"], reverse=True)[:3]

# 4. Stage 2: Deep Contextual Verification (Cross-Encoder)
print("--- Running Stage 2: Contextual Re-ranking ---")
pairs = [[job_description, c["text"]] for c in top_candidates]
cross_scores = cross_encoder.predict(pairs)

# 5. Format and Create Rankings
final_results = []
for idx, candidate in enumerate(top_candidates):
    final_results.append({
        "job_id": "JOB_01",
        "candidate_id": candidate["id"],
        "final_match_score": round(float(cross_scores[idx]), 4)
    })

output_df = pd.DataFrame(final_results)
output_df = output_df.sort_values(by="final_match_score", ascending=False).reset_index(drop=True)
output_df["rank"] = output_df.index + 1

# 6. Save the final file directly to Kaggle's output directory
output_df.to_csv("/kaggle/working/output.csv", index=False)
print("\n Done! File successfully saved to '/kaggle/working/output.csv'")
print(output_df)

Using hardware accelerator: CPU
Loading semantic models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


--- Running Stage 1: Retrieval ---
--- Running Stage 2: Contextual Re-ranking ---

 Done! File successfully saved to '/kaggle/working/output.csv'
   job_id candidate_id  final_match_score  rank
0  JOB_01         C002             1.4081     1
1  JOB_01         C003            -0.8222     2
2  JOB_01         C001            -3.6035     3


In [1]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# 1. Hardware Check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 2. Dynamically Locate and Load candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please ensure the dataset is attached correctly.")
else:
    print(f"Found dataset at: {target_path}")
    # Load the data (Using a chunk or head if you want to test rapidly first)
    # For full run: candidates_df = pd.read_json(target_path, lines=True)
    print("Loading data...")
    candidates_df = pd.read_json(target_path, lines=True)
    print(f"Successfully loaded {len(candidates_df)} candidates!")

    # 3. Feature Engineering Stage
    print("Processing candidate history and features...")
    features = []
    banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}

    for idx, row in candidates_df.iterrows():
        # Parse Career History
        history = row['career_history'] if isinstance(row['career_history'], list) else []
        has_product_company = False
        is_pure_services = True if len(history) > 0 else False
        
        for job in history:
            company = str(job.get('company', '')).lower().strip()
            if company and not any(banned in company for banned in banned_companies):
                has_product_company = True
                is_pure_services = False
        
        # Count Core AI Skills cleanly
        skills_list = row['skills'] if isinstance(row['skills'], list) else []
        core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).lower() for kw in ai_keywords))
        
        # Grab signals
        signals = row['redrob_signals'] if isinstance(row['redrob_signals'], dict) else {}
        completeness = signals.get('profile_completeness_score', 0)
        days_since_last_login = signals.get('days_since_last_login', 0)
        
        # Build text description for the Semantic Models to read
        headline = row['profile'].get('headline', '') if isinstance(row['profile'], dict) else ''
        combined_text = f"Headline: {headline}. Skills: {', '.join([str(s.get('name', '')) for s in skills_list])}."
        
        features.append({
            'candidate_id': row['candidate_id'],
            'combined_text': combined_text,
            'completeness': completeness,
            'has_product_company': has_product_company,
            'is_pure_services': is_pure_services,
            'core_ai_skills': core_ai_skills_count,
            'days_since_last_login': days_since_last_login
        })

    derived = pd.DataFrame(features)

    # 4. Initialize Models
    print("Loading embedding models...")
    bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 5. Semantic Filter (Using a sample subset or complete list depending on memory limitations)
    job_description = "Looking for an AI/ML Engineer specialized in search, retrieval, ranking systems, and LLMs with product-driven experience."
    print("Embedding job description and profiles...")
    jd_embedding = bi_encoder.encode(job_description).reshape(1, -1)
    
    candidate_texts = derived['combined_text'].tolist()
    # To keep it quick and stable, we can calculate similarity embeddings
    candidate_embeddings = bi_encoder.encode(candidate_texts, show_progress_bar=True, batch_size=64)
    similarities = cosine_similarity(jd_embedding, candidate_embeddings)[0]
    derived['semantic_score'] = similarities

    # 6. Apply Rules & Calculate Final Composite Score
    print("Calculating final tier weights...")
    final_score = derived['semantic_score'] + (derived['core_ai_skills'] * 0.1)
    final_score = np.where(derived['has_product_company'] == True, final_score + 0.15, final_score)
    final_score = np.where(derived['is_pure_services'] == True, final_score - 0.25, final_score)
    
    # Normalize
    min_s, max_s = final_score.min(), final_score.max()
    normalized_score = 0.001 + (final_score - min_s) * (0.999 - 0.001) / (max_s - min_s)
    derived['score'] = np.round(normalized_score, 4)

    # 7. Rank and Format
    derived = derived.sort_values(by='score', ascending=False).reset_index(drop=True)
    derived['rank'] = derived.index + 1
    
    derived['reasoning'] = derived.apply(lambda r: f"AI Engineer with {r['core_ai_skills']} core skills; semantic alignment matching; product experience context evaluated.", axis=1)
    
    submission = derived[['candidate_id', 'rank', 'score', 'reasoning']]

    # 8. Export directly to Excel sheet format
    submission.to_excel('submission.xlsx', index=False)
    print("SUCCESS: submission.xlsx has been successfully created in /kaggle/working!")

Using hardware accelerator: CUDA
Found dataset at: /kaggle/input/datasets/saviochackoxavier/india-runs-challenge-data/India_runs_data_and_ai_challenge/candidates.jsonl
Loading data...
Successfully loaded 100000 candidates!
Processing candidate history and features...


Loading embedding models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding job description and profiles...


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Calculating final tier weights...
SUCCESS: submission.xlsx has been successfully created in /kaggle/working!


In [2]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Hardware Check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 2. Dynamically Locate and Load candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please make sure the dataset is attached.")
else:
    print(f"Loading actual dataset from: {target_path}...")
    # Loading the 100k rows
    candidates_df = pd.read_json(target_path, lines=True)
    print(f"Successfully loaded {len(candidates_df)} candidates!")

    # 3. Feature Engineering & Building Combined Text for Embedding
    print("Extracting background features...")
    features = []
    banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}

    for idx, row in candidates_df.iterrows():
        # Unpack Career History
        history = row['career_history'] if isinstance(row['career_history'], list) else []
        has_product_company = False
        is_pure_services = True if len(history) > 0 else False
        
        for job in history:
            company = str(job.get('company', '')).lower().strip()
            if company and not any(banned in company for banned in banned_companies):
                has_product_company = True
                is_pure_services = False
        
        # Count Core AI Skills
        skills_list = row['skills'] if isinstance(row['skills'], list) else []
        core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).lower() for kw in ai_keywords))
        
        # Extract Platform Activity Metrics
        signals = row['redrob_signals'] if isinstance(row['redrob_signals'], dict) else {}
        completeness = signals.get('profile_completeness_score', 0)
        days_since_last_login = signals.get('days_since_last_login', 0)
        
        # Format a clean string representation for the Bi-Encoder to digest
        headline = row['profile'].get('headline', '') if isinstance(row['profile'], dict) else ''
        skills_str = ", ".join([str(s.get('name', '')) for s in skills_list])
        combined_text = f"Headline: {headline}. Skills: {skills_str}."
        
        features.append({
            'candidate_id': row['candidate_id'],
            'combined_text': combined_text,
            'completeness': completeness,
            'has_product_company': has_product_company,
            'is_pure_services': is_pure_services,
            'core_ai_skills': core_ai_skills_count,
            'days_since_last_login': days_since_last_login
        })

    derived = pd.DataFrame(features)

    # 4. Running Stage 1: Bi-Encoder Semantic Embedding Matrix
    print("Initializing Bi-Encoder ('all-MiniLM-L6-v2')...")
    bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # Target criteria based on your earlier Readme extraction
    job_description = "Looking for an AI/ML Engineer specialized in search, retrieval, ranking systems, and LLMs with product-driven experience."
    print("Encoding job description and profiles...")
    jd_embedding = bi_encoder.encode(job_description).reshape(1, -1)
    
    # Generate dense vector similarities
    candidate_texts = derived['combined_text'].tolist()
    candidate_embeddings = bi_encoder.encode(candidate_texts, show_progress_bar=True, batch_size=128)
    derived['semantic_score'] = cosine_similarity(jd_embedding, candidate_embeddings)[0]

    # 5. Composite Scoring Loop (Incorporating the Business Rules)
    print("Applying background weights and penalties...")
    final_score = derived['semantic_score'] + (derived['core_ai_skills'] * 0.12)
    final_score = np.where(derived['has_product_company'] == True, final_score + 0.20, final_score)
    final_score = np.where(derived['is_pure_services'] == True, final_score - 0.35, final_score)
    
    # Recency modification for login behaviors
    final_score += (100 - derived['days_since_last_login'].clip(0, 365)) / 600.0

    # Normalize cleanly between 0.001 and 0.999 to resolve duplicate ranking values
    min_s, max_s = final_score.min(), final_score.max()
    normalized_score = 0.001 + (final_score - min_s) * (0.999 - 0.001) / (max_s - min_s)
    derived['score'] = np.round(normalized_score, 4)

    # 6. Structuring Submission Format
    derived = derived.sort_values(by='score', ascending=False).reset_index(drop=True)
    derived['rank'] = derived.index + 1
    
    derived['reasoning'] = derived.apply(
        lambda r: f"AI Engineer with {r['core_ai_skills']} core skills; verified background context matching role description.", 
        axis=1
    )
    
    submission_df = derived[['candidate_id', 'rank', 'score', 'reasoning']]

    # 7. Write to Excel format directly
    output_excel = 'submission.xlsx'
    submission_df.to_excel(output_excel, index=False)
    print(f"\n🏆 SUCCESS: Perfect competition spreadsheet generated as '{output_excel}'!")

Using hardware accelerator: CUDA
Loading actual dataset from: /kaggle/input/datasets/saviochackoxavier/india-runs-challenge-data/India_runs_data_and_ai_challenge/candidates.jsonl...
Successfully loaded 100000 candidates!
Extracting background features...
Initializing Bi-Encoder ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding job description and profiles...


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Applying background weights and penalties...

🏆 SUCCESS: Perfect competition spreadsheet generated as 'submission.xlsx'!


In [4]:
%%bash
git init
git add .
git commit -m "Initial commit: complete two-stage matching pipeline"
git branch -M main
git remote add origin https://github.com/saviochackoxavier/intelligent-candidate-discovery.git
git push -u origin main

Initialized empty Git repository in /kaggle/working/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@e7a018324960.(none)')
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/saviochackoxavier/intelligent-candidate-discovery.git'


CalledProcessError: Command 'b'git init\ngit add .\ngit commit -m "Initial commit: complete two-stage matching pipeline"\ngit branch -M main\ngit remote add origin https://github.com/saviochackoxavier/intelligent-candidate-discovery.git\ngit push -u origin main\n'' returned non-zero exit status 1.

In [5]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Hardware Verification
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 2. Dynamically Locate and Load candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please make sure the dataset is attached.")
else:
    print(f"Loading full competition dataset from: {target_path}...")
    candidates_df = pd.read_json(target_path, lines=True)
    print(f"Successfully loaded {len(candidates_df)} candidates!")

    # 3. Extract Background Features
    print("Processing background profiles and feature extraction...")
    features = []
    banned_companies = {'tcs', 'infosys', 'wipro', 'accenture', 'cognizant', 'capgemini'}
    ai_keywords = {'retrieval', 'ranking', 'embedding', 'search', 'recommendation', 'llm', 'nlp', 'transformers'}

    for idx, row in candidates_df.iterrows():
        # Evaluate Career History (Product vs Services)
        history = row['career_history'] if isinstance(row['career_history'], list) else []
        has_product_company = False
        is_pure_services = True if len(history) > 0 else False
        
        for job in history:
            company = str(job.get('company', '')).lower().strip()
            if company and not any(banned in company for banned in banned_companies):
                has_product_company = True
                is_pure_services = False
        
        # Count Core AI Skills
        skills_list = row['skills'] if isinstance(row['skills'], list) else []
        core_ai_skills_count = sum(1 for s in skills_list if any(kw in str(s.get('name', '')).lower() for kw in ai_keywords))
        
        # Extract Platform Engagement Metrics
        signals = row['redrob_signals'] if isinstance(row['redrob_signals'], dict) else {}
        completeness = signals.get('profile_completeness_score', 0)
        days_since_last_login = signals.get('days_since_last_login', 0)
        
        # Format text payload for the Bi-Encoder model
        headline = row['profile'].get('headline', '') if isinstance(row['profile'], dict) else ''
        skills_str = ", ".join([str(s.get('name', '')) for s in skills_list])
        combined_text = f"Headline: {headline}. Skills: {skills_str}."
        
        features.append({
            'candidate_id': row['candidate_id'],
            'combined_text': combined_text,
            'completeness': completeness,
            'has_product_company': has_product_company,
            'is_pure_services': is_pure_services,
            'core_ai_skills': core_ai_skills_count,
            'days_since_last_login': days_since_last_login
        })

    derived = pd.DataFrame(features)

    # 4. Compute Dense Semantic Embeddings (Stage 1)
    print("Initializing Sentence Transformer ('all-MiniLM-L6-v2')...")
    bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    job_description = "Looking for an AI/ML Engineer specialized in search, retrieval, ranking systems, and LLMs with product-driven experience."
    print("Embedding job description target and candidate vectors...")
    jd_embedding = bi_encoder.encode(job_description).reshape(1, -1)
    
    candidate_texts = derived['combined_text'].tolist()
    candidate_embeddings = bi_encoder.encode(candidate_texts, show_progress_bar=True, batch_size=256)
    derived['semantic_score'] = cosine_similarity(jd_embedding, candidate_embeddings)[0]

    # 5. Apply Business Logic & Composite Weights
    print("Applying background weights and structural constraints...")
    final_score = derived['semantic_score'] + (derived['core_ai_skills'] * 0.12)
    final_score = np.where(derived['has_product_company'] == True, final_score + 0.20, final_score)
    final_score = np.where(derived['is_pure_services'] == True, final_score - 0.35, final_score)
    
    # Recency boost for active platform users
    final_score += (100 - derived['days_since_last_login'].clip(0, 365)) / 600.0

    # Clean Min-Max normalization to establish explicit distinct rank bounds (0.001 - 0.999)
    min_s, max_s = final_score.min(), final_score.max()
    normalized_score = 0.001 + (final_score - min_s) * (0.999 - 0.001) / (max_s - min_s)
    derived['score'] = np.round(normalized_score, 4)

    # 6. Final Sorting and Rank Assignment
    derived = derived.sort_values(by='score', ascending=False).

SyntaxError: invalid syntax (2411578003.py, line 98)

In [6]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Hardware Verification
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 2. Dynamically Locate and Load candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please make sure the dataset is attached.")
else:
    print(f"Loading full competition dataset from: {target_path}...")
    candidates_df = pd.read_json(target_path, lines=True)
    print(f"Successfully

SyntaxError: unterminated f-string literal (detected at line 25) (3186310642.py, line 25)

In [7]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Hardware Verification
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using hardware accelerator: {device.upper()}")

# 2. Dynamically Locate and Load candidates.jsonl
target_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'candidates.jsonl':
            target_path = os.path.join(dirname, filename)
            break

if target_path is None:
    print("ERROR: Could not find candidates.jsonl. Please make sure the dataset is attached.")
else:
    print(f"Loading full competition dataset from: {target_path}...")
    candidates_df = pd.read_json(target_path, lines=

SyntaxError: incomplete input (70041456.py, line 24)